## Simulate 10 new machine observations
This is just a basic check for the pipeline

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Original dataset
landing_path = (
    "/Volumes/predictivemaintenancecatalog/"
    "raw_data/landing/testdata"
)

# Registered table monitored by Lakeflow
landing_table = (
    "predictivemaintenancecatalog.raw_data.landing_testdata"
)

# STEP 1: Create a batch of 10 observations
new_batch = (
    spark.read.format("delta")
    .load(landing_path)
    .orderBy(F.rand())
    .limit(10)
)

# STEP 2: Find the maximum UDI in the registered table
max_udi = (
    spark.table(landing_table)
    .agg(F.max("UDI"))
    .first()[0]
)

# STEP 3: Assign new unique UDIs
next_batch = new_batch.withColumn(
    "UDI",
    (
        F.lit(max_udi if max_udi is not None else 0)
        + F.row_number().over(Window.orderBy("UDI"))
    ).cast(new_batch.schema["UDI"].dataType)
)

# STEP 4: Append to the registered Delta table
(
    next_batch.write
    .format("delta")
    .mode("append")
    .option(
        "userMetadata",
        "Simulated batch of 10 machines for Lakeflow streaming test"
    )
    .saveAsTable(landing_table)
)

print("10 new machine observations added.")

In [0]:
spark.sql(f"DESCRIBE HISTORY {landing_table}") \
    .select("version", "timestamp", "operation", "userMetadata") \
    .show(truncate=False)

## Bad Data and Data Quality Validation
this section is just to simulate if the pipeline will work if the dataset is bad.
I will see if the quality check rules for the silver table works

In [0]:
from pyspark.sql import functions as F

landing_table = (
    "predictivemaintenancecatalog.raw_data.landing_testdata"
)

# Use the registered table for generating the new batch
landing_df = spark.table(landing_table)

# Check maximum UDI across existing datasets
sources = [
    landing_df,
    spark.read.format("delta").load(landing_path),
    spark.table(
        "predictivemaintenancecatalog."
        "bronze_data_not_quality_checked.streaming"
    ),
    spark.table(
        "predictivemaintenancecatalog.silver_data.streaming"
    )
]

max_udi = max(
    int(value)
    for df in sources
    if (value := df.agg(F.max("UDI")).first()[0]) is not None
)

print("Highest existing UDI:", max_udi)


# Select only originally valid records
valid_filter = (
    F.col("UDI").isNotNull()
    & F.upper(F.trim(F.col("Type"))).isin("L", "M", "H")
    & (F.col("Air_temperature_K") > 0)
    & (F.col("Process_temperature_K") > 0)
    & (F.col("Rotational_speed_rpm") > 0)
    & (F.col("Torque_Nm") >= 0)
    & (F.col("Tool_wear_min") >= 0)
)

seed_rows = (
    landing_df
    .filter(valid_filter)
    .orderBy("UDI")
    .limit(8)
    .collect()
)

assert len(seed_rows) == 8

# Assign new UDIs

test_rows = []

for i, row in enumerate(seed_rows, start=1):

    record = row.asDict()
    record["UDI"] = int(max_udi) + i

    if i == 1:
        record["Type"] = "X"
    elif i == 2:
        record["Air_temperature_K"] = -5.0
    elif i == 3:
        record["Process_temperature_K"] = 0.0
    elif i == 4:
        record["Rotational_speed_rpm"] = 0
    elif i == 5:
        record["Torque_Nm"] = -10.0
    elif i == 6:
        record["Tool_wear_min"] = -1

    test_rows.append(record)

bad_batch = spark.createDataFrame(
    test_rows,
    schema=landing_df.schema
)

# Generate test IDs BEFORE checking for duplicates
test_ids = [r["UDI"] for r in test_rows]

# Confirm all 8 IDs are unique
assert len(set(test_ids)) == 8

# Check that none of the IDs already exist
for df in sources:
    duplicates = (
        df.filter(F.col("UDI").isin(test_ids))
        .count()
    )

    assert duplicates == 0, (
        f"Found {duplicates} existing records "
        "with these UDIs!"
    )

print("All 8 UDIs are unique across checked datasets.")

display(bad_batch)
print("Test UDIs:", test_ids)

In [0]:
(
    bad_batch.write
    .format("delta")
    .mode("append")
    .option(
        "userMetadata",
        "Data quality test: 6 invalid, 2 valid"
    )
    .saveAsTable(landing_table)
)

print("8 new machine observations added.")